# AutoScout AI – trénování modelu
Tento notebook popisuje postup načtení vlastního datasetu, jeho vyčištění, trénování modelu a vyhodnocení.

## 1. Nahraj vlastní `autos_raw.csv` do Colabu
Použij pouze data získaná vlastním crawlerem.

In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('autos_raw.csv')
df.head()

## 2. Vyčištění dat
Stejná logika je i v souboru `preprocess.py` v projektu.

In [ ]:
import re
from datetime import datetime
CURRENT_YEAR = datetime.now().year
def parse_int(value):
    if pd.isna(value):
        return None
    digits = re.sub(r'[^\d]', '', str(value))
    return int(digits) if digits else None
for col in ['price_czk','year','km','power_kw','engine_ccm','doors','seats','owners_count']:
    if col in df.columns:
        df[col] = df[col].apply(parse_int)
df = df.dropna(subset=['price_czk','year','km','brand','model'])
df = df[df['price_czk'].between(10000, 5000000)]
df = df[df['year'].between(1990, CURRENT_YEAR + 1)]
df = df[df['km'].between(0, 600000)]
df['age'] = CURRENT_YEAR - df['year']
df['km_per_year'] = df['km'] / df['age'].replace(0, 1)
df.head()

## 3. Trénování modelu
Použit je regresní model RandomForestRegressor.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features = ['brand','model','fuel','transmission','body_type','year','age','km','km_per_year','power_kw','engine_ccm','doors','seats','owners_count']
target = 'price_czk'
X = df[features].copy()
y = df[target].copy()

num = ['year','age','km','km_per_year','power_kw','engine_ccm','doors','seats','owners_count']
cat = ['brand','model','fuel','transmission','body_type']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat),
] )
model = RandomForestRegressor(n_estimators=250, max_depth=18, min_samples_split=4, min_samples_leaf=2, random_state=42, n_jobs=-1)
pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)
print('MAE:', mean_absolute_error(y_test, preds))
print('RMSE:', mean_squared_error(y_test, preds, squared=False))
print('R2:', r2_score(y_test, preds))

## 4. Export modelu
Model se v lokálním projektu ukládá pomocí `joblib` v souboru `model_bundle.joblib`.
